# 20.6 Batch vs online inference

Batch inference buys cheap scale with stale predictions; online inference buys freshness with latency and reliability pressure.

Feature freshness defines the staleness term, while system design defines latency and cost budgets. The notebook keeps those quantities inspectable across increasingly realistic serving workloads.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 2006
rng = np.random.default_rng(SEED)


def hybrid_latency(hit_rate, cache_ms, online_ms):
    return hit_rate * cache_ms + (1.0 - hit_rate) * online_ms


def batch_online_ladder(seed=SEED):
    local_rng = np.random.default_rng(seed)
    specs = [
        {"rung": "D1", "name": "10 request cache trace", "n": 10, "hit_rate": 0.70, "qps": 20, "refresh_hours": 6},
        {"rung": "D2", "name": "100k clean requests", "n": 100000, "hit_rate": 0.82, "qps": 180, "refresh_hours": 6},
        {"rung": "D3", "name": "peak misses and concurrency", "n": 60000, "hit_rate": 0.58, "qps": 500, "refresh_hours": 3},
        {"rung": "D4", "name": "diurnal request trace", "n": 86400, "hit_rate": 0.72, "qps": 260, "refresh_hours": 2},
        {"rung": "D5", "name": "production mixed serving", "n": 250000, "hit_rate": 0.64, "qps": 900, "refresh_hours": 1},
    ]
    workloads = []
    for spec in specs:
        n = spec["n"]
        t = np.arange(n) / max(spec["qps"], 1)
        diurnal = 1.0 + 0.35 * np.sin(2.0 * np.pi * t / max(t[-1], 1.0))
        burst = local_rng.gamma(shape=2.0, scale=1.0, size=n)
        load = spec["qps"] * diurnal * (0.85 + 0.15 * burst)
        hit_probability = np.clip(spec["hit_rate"] - 0.10 * (load > np.percentile(load, 90)), 0.05, 0.98)
        hits = local_rng.random(n) < hit_probability
        workloads.append({**spec, "load": load, "hits": hits})
    return workloads


def simulate_batch_online(workload, online_ms=80.0, cache_ms=5.0):
    load = workload["load"]
    hits = workload["hits"]
    miss_pressure = np.maximum(load - 500.0, 0.0) / 500.0
    queue_ms = 18.0 * miss_pressure * (~hits)
    jitter_ms = rng.normal(0.0, 2.0, size=hits.size)
    latency = np.where(hits, cache_ms, online_ms) + queue_ms + np.maximum(jitter_ms, -2.0)
    latency = np.maximum(latency, 1.0)
    cost = hits.size * 0.00002 + np.sum(~hits) * 0.00008
    throughput = hits.size / max(hits.size / max(np.mean(load), 1.0), 1.0)
    staleness_hours = np.where(hits, workload["refresh_hours"] / 2.0, 0.0)
    return {"latency_ms": latency, "cost": cost, "throughput_qps": throughput, "staleness_hours": staleness_hours}


## The concept, built once: name the online path

The lesson's online latency is the sum of live feature lookup, model execution, and network time: $25+40+15=80$ ms. Batch staleness for a 6-hour refresh has worst-case age $6$ hours and average age $3$ hours.

In [ ]:

feature_lookup_ms = 25
model_ms = 40
network_ms = 15
online_ms = feature_lookup_ms + model_ms + network_ms
worst_case_age_hours = 6
average_age_hours = worst_case_age_hours / 2

assert online_ms == 80
assert average_age_hours == 3
print("online path ms", online_ms)
print("average batch age hours", average_age_hours)


Now combine cached and online traffic with $E[L]=hL_{cache}+(1-h)L_{online}$. The same notation gives expected latency, peak concurrency, and batch cost before any large model is involved.

In [ ]:

hit_rate = 0.70
cache_ms = 5
expected_ms = hybrid_latency(hit_rate, cache_ms, online_ms)
batch_predictions = 1_000_000
batch_unit_cost = 0.00002
batch_cost = batch_predictions * batch_unit_cost
peak_qps = 500
in_flight = peak_qps * (online_ms / 1000.0)

assert expected_ms == 27.5
assert batch_cost == 20.0
assert in_flight == 40.0
print("hybrid expected latency ms", expected_ms)
print("batch cost dollars", batch_cost)
print("online in-flight requests", in_flight)


## The dataset ladder
D1-D5 move from a 10-request toy trace to a production-scale mixed batch/online simulation.

In [ ]:

workloads = batch_online_ladder()
for workload in workloads:
    preview = pd.DataFrame({
        "hit": workload["hits"][:8],
        "load_qps": np.round(workload["load"][:8], 1),
    })
    print(workload["rung"], workload["name"], "n=", workload["n"], "target hit=", workload["hit_rate"])
    print(preview)


In [ ]:

rows = []
outputs = []
for workload in workloads:
    result = simulate_batch_online(workload)
    outputs.append((workload, result))
    p95 = float(np.percentile(result["latency_ms"], 95))
    rows.append({
        "rung": workload["rung"],
        "workload": workload["name"],
        "metric_p95_latency_ms": p95,
        "mean_staleness_hours": float(np.mean(result["staleness_hours"])),
        "cost_dollars": float(result["cost"]),
        "throughput_qps": float(result["throughput_qps"]),
    })
metrics = pd.DataFrame(rows)
print(metrics.to_string(index=False))


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for i, (workload, result) in enumerate(outputs):
    axes[0, i].hist(result["latency_ms"], bins=35, color="#4c78a8")
    axes[0, i].set_title(workload["rung"])
    axes[0, i].set_xlabel("latency ms")
    axes[0, i].set_ylabel("requests")
    panel_values = [np.mean(result["latency_ms"]), np.percentile(result["latency_ms"], 95), result["throughput_qps"]]
    axes[1, i].bar(["mean", "p95", "qps"], panel_values, color=["#72b7b2", "#f58518", "#54a24b"])
    axes[1, i].set_title("operational panel")
summary_x = np.arange(len(metrics))
fig2, ax = plt.subplots(figsize=(7, 4))
ax.plot(summary_x, metrics["metric_p95_latency_ms"], marker="o")
ax.set_xticks(summary_x)
ax.set_xticklabels(metrics["rung"])
ax.set_ylabel("p95 latency ms")
ax.set_xlabel("D1 to D5 load")
ax.set_title("metric vs load")
plt.show()


## Pitfall on D5: optimizing freshness without latency
A pure online route has zero staleness, but it pushes every request through the expensive path. A hybrid cache can keep most user-visible latency low while only refreshing misses online.

In [ ]:

d5 = workloads[-1]
all_online = {**d5, "hits": np.zeros(d5["n"], dtype=bool)}
hybrid = d5
all_online_result = simulate_batch_online(all_online)
hybrid_result = simulate_batch_online(hybrid)
wrong_p95 = float(np.percentile(all_online_result["latency_ms"], 95))
fixed_p95 = float(np.percentile(hybrid_result["latency_ms"], 95))
improvement = wrong_p95 - fixed_p95
print("all-online p95 ms", wrong_p95)
print("hybrid-routed p95 ms", fixed_p95)
print("p95 improvement ms", improvement)


## Evaluate it + practice

- Metric: p95 user-visible latency, with cost and staleness as guardrails.
- No-skill baseline: send everything online and accept the tail latency.
- Cheap sanity check: if hit_rate=1, p95 should be near cache latency.
- Ablation: turn off cache routing and p95 rises on peak workloads.
- Failure signals: low mean latency with high p95, hidden concurrency, or stale cached scores.

Practice: Change D5 hit rate from 0.64 to 0.80 and predict the p95 movement.

Practice: Add a second cache tier with 15 ms latency.

Practice: Plot cost against p95 latency for every rung.